# Clase 6: Árboles de decisión y ensamblaje. Visualizaciones

Cada celda muestra una visualización con controles. Todo el código de dibujo está en `viz_arboles.py`, en esta misma carpeta.

Requiere `numpy`, `matplotlib`, `scikit-learn`, `ipywidgets` y `xgboost` (solo la última visualización).

In [1]:
%matplotlib inline
import viz_arboles as viz

## 1. Tú eliges la pregunta

Datos del esquí: mes y latitud, positivo si se puede esquiar. Escoge la feature y el umbral de la **primera** pregunta del árbol. A la izquierda se ven los dos hijos con sus conteos. A la derecha, la ganancia de **todos** los umbrales posibles de esa feature, con las tres medidas de impureza.

**Qué observar**
- La pregunta "natural" de la clase, latitud $< 30$, tiene una ganancia pequeña. El algoritmo codicioso preferiría otra. ¿Cuál?
- El error de clasificación (línea gris) vale cero en casi todos los umbrales: no distingue entre preguntas.
- Gini y entropía tienen la misma forma y sus máximos están en el mismo lugar.

In [2]:
viz.explorar_division()

Output()

## 2. El árbol nivel por nivel

El árbol que construye scikit-learn sobre los mismos datos, con profundidad máxima ajustable.

**Qué observar**
- Cada nivel agrega cortes paralelos a los ejes, y las regiones son rectángulos.
- Con profundidad suficiente la exactitud de entrenamiento llega a 100%.
- Cambiar `gini` por `entropy` casi nunca cambia el árbol.

In [3]:
viz.arbol_por_niveles()

interactive(children=(IntSlider(value=2, continuous_update=False, description='profundidad máxima', max=8, min…

## 3. Concavidad: por qué Gini funciona y el error de clasificación no

Mueve las proporciones de positivos de los dos hijos ($\hat p_1$, $\hat p_2$) y el tamaño relativo del primero. El padre está en el promedio ponderado $\hat p_p$. La ganancia (en rojo) es la distancia entre la curva y la cuerda.

**Qué observar**
- Con Gini o entropía, la ganancia es positiva siempre que $\hat p_1 \neq \hat p_2$.
- Con el error de clasificación, si los dos hijos quedan del mismo lado de $0.5$, la ganancia es exactamente 0.
- La ganancia crece cuanto más separados están $\hat p_1$ y $\hat p_2$.

In [5]:
viz.concavidad()

interactive(children=(FloatSlider(value=0.6, continuous_update=False, description='p̂₁', max=1.0, step=0.01, s…

## 4. Árboles de regresión

Nieve por día según el mes. Cada hoja predice el promedio de su región.

**Qué observar**
- La predicción es una escalera, con un escalón por hoja.
- El error de entrenamiento siempre baja con la profundidad. El de prueba baja y después sube un poco: overfitting.
- `min_samples_leaf` regulariza: con valores grandes, el árbol no puede seguir puntos individuales.

In [ ]:
viz.arbol_regresion()

interactive(children=(IntSlider(value=2, continuous_update=False, description='max_depth', max=12, min=1, styl…

## 5. Overfitting y regularización

Datos ruidosos en dos dimensiones. Compara tres formas de regularizar un árbol: profundidad máxima, tamaño mínimo de hoja y poda (`ccp_alpha`).

**Qué observar**
- Sin regularización: 100% en entrenamiento, alrededor de 79% en prueba, y una frontera llena de "islas" que siguen el ruido.
- Con un poco de poda (`ccp_alpha` ≈ 0.01) el árbol queda con pocas hojas y mejora en prueba.

In [5]:
viz.regularizacion()

interactive(children=(IntSlider(value=12, continuous_update=False, description='max_depth', max=15, min=1, sty…

## 6. Varianza del promedio de modelos correlacionados

Si promediamos $B$ modelos con varianza $\sigma^2 = 1$ y correlación $\rho$ entre cada par:

$$\text{Var}(\bar X) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2.$$

Los puntos rojos son una simulación, y el histograma compara las predicciones de un modelo con las del promedio.

**Qué observar**
- Con $\rho = 0$ la varianza tiende a cero. Con $\rho > 0$ se estanca en $\rho\sigma^2$, por grande que sea $B$.
- Por eso random forests trata de bajar $\rho$.

In [6]:
viz.varianza_promedio()

interactive(children=(FloatSlider(value=0.3, continuous_update=False, description='ρ (correlación)', max=0.95,…

## 7. Bagging y random forest

Tres de los árboles del ensamblaje y la predicción promediada de todos. Cada árbol se entrena con una muestra bootstrap. En **random forest**, además, cada división solo puede usar una feature sorteada al azar.

**Qué observar**
- Los árboles individuales son muy distintos entre sí y todos sobreajustan (cerca de 80% en prueba).
- El promedio tiene una frontera más suave y mejor exactitud de prueba.
- Al aumentar $B$ la exactitud se estabiliza: no hay overfitting por usar más árboles.
- Cambia la semilla: los árboles individuales cambian mucho, el promedio casi nada.

In [ ]:
viz.bagging()

interactive(children=(IntSlider(value=50, continuous_update=False, description='B', max=300, min=1, style=Slid…

## 8. AdaBoost paso a paso

A la izquierda, el tocón de la ronda $b$, entrenado con los pesos actuales: el tamaño de cada punto es proporcional a su peso. A la derecha, la votación ponderada $\text{signo}(\sum_b \alpha_b G_b(x))$ de los tocones hasta esa ronda.

**Qué observar**
- Los ejemplos mal clasificados en una ronda crecen en la siguiente.
- Con tres tocones la votación ya clasifica todo bien, con una frontera que ningún tocón podía dibujar solo.

In [8]:
viz.adaboost()

interactive(children=(IntSlider(value=1, description='ronda b', max=15, min=1), Output()), _dom_classes=('widg…

## 9. Gradient boosting: ajustar los residuos

Boosting con pérdida cuadrática sobre los datos de nieve: $F_b = F_{b-1} + \nu h_b$, donde $h_b$ es un árbol entrenado con los residuos $r_i = y^{(i)} - F_{b-1}(x^{(i)})$.

**Qué observar**
- Con $B = 0$ el modelo es el promedio, y los residuos son los datos "centrados".
- Cada árbol nuevo ajusta lo que todavía falta explicar (panel del centro).
- En el panel derecho, el error de prueba llega a un mínimo y luego sube: en boosting, aumentar $B$ **sí** puede producir overfitting.
- Con $\nu$ más pequeño, hacen falta más árboles, pero el mínimo suele ser mejor.

In [11]:
viz.gradient_boosting()

interactive(children=(IntSlider(value=1, description='B', max=200), FloatSlider(value=0.3, continuous_update=F…

## 10. XGBoost: $\lambda$ encoge las hojas y $\gamma$ poda

Un solo árbol de XGBoost (con la librería `xgboost`) ajustado a los residuos respecto al promedio. La tabla muestra, para cada hoja, el número de ejemplos $n_j$, el promedio de los residuos, el valor que XGBoost le asignó a la hoja y la fórmula de las notas, $S_j/(n_j + \lambda)$, donde $S_j$ es la suma de los residuos de la hoja.

**Qué observar**
- Con $\lambda = 0$ cada hoja vale el promedio de sus residuos, como un árbol de regresión normal.
- Al subir $\lambda$, las hojas se encogen hacia cero, y mucho más las que tienen pocos ejemplos.
- La columna de XGBoost coincide exactamente con la fórmula.
- Al subir $\gamma$ desaparecen las divisiones con poca ganancia y el árbol tiene menos hojas.

In [10]:
viz.xgboost_regularizacion()

interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='λ (reg_lambda)', max=50.0, …